### we can train a simple model to classify

In [3]:
# !pip install transformers datasets torch scikit-learn pandas evaluate

In [4]:
import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
WELCOME = [
    "Greeting the candidate at the beginning of an interview.",
    "Welcoming the candidate and establishing a friendly atmosphere.",
    "Asking how the candidate is doing today.",
    "Thanking the candidate for joining the interview.",
    "Introducing the interviewer and welcoming the candidate.",
    "Good morning, how are you doing today?",
    "Hello, it's nice to meet you.",
    "Thank you for taking the time to meet with us."
]

WARM_UP = [
    "Asking the candidate to introduce themselves.",
    "Understanding the candidate's personal and educational background.",
    "Learning about the candidate's interests and career aspirations.",
    "Understanding why the candidate applied for this position.",
    "Exploring the candidate's motivation and career goals.",
    "What are your greatest strengths?",
    "Why are you interested in this role?",
    "How would you describe yourself professionally?",
    "What motivates you in your career?",
    "What are you looking for in your next opportunity?"
]

EXPERIENCE = [
    "Exploring the candidate's previous professional experience.",
    "Understanding previous jobs, internships and professional responsibilities.",
    "Discussing projects the candidate has completed.",
    "Understanding the candidate's individual contributions to projects.",
    "Evaluating practical experience and ownership of responsibilities.",
    "What projects have you worked on?",
    "What was your role in your previous project?",
    "What responsibilities did you handle during your internship?",
    "What is your most significant professional achievement?",
    "What was the biggest challenge you faced while developing your project?",
    "Which technologies have you used in your previous work?"
]

BEHAVIORAL = [
    "Exploring how the candidate behaved in previous real-life situations.",
    "Evaluating teamwork and collaboration through past experiences.",
    "Understanding how the candidate handled workplace conflicts.",
    "Evaluating leadership and communication through previous examples.",
    "Understanding how the candidate responded to failure and criticism.",
    "Asking about situations involving pressure and tight deadlines.",
    "Tell me about a time you disagreed with a colleague.",
    "Describe a situation where you helped a struggling teammate.",
    "Tell me about a mistake you made and how you handled it.",
    "Describe a time you received negative feedback.",
    "Tell me about a time you had to take responsibility for a failure.",
    "Describe a situation where you demonstrated leadership."
]

SITUATIONAL = [
    "Presenting hypothetical workplace situations to evaluate decision-making.",
    "Evaluating how the candidate would respond to unexpected challenges.",
    "Understanding how the candidate would resolve hypothetical conflicts.",
    "Assessing problem-solving and adaptability in unfamiliar situations.",
    "Evaluating prioritization when facing competing responsibilities.",
    "What would you do if your teammate refused to cooperate?",
    "How would you handle multiple urgent deadlines?",
    "Imagine you are assigned a task you have never performed before. What would you do?",
    "What would you do if you disagreed with your manager?",
    "How would you respond if a project suddenly failed?",
    "Suppose you discovered a serious mistake just before delivery. How would you handle it?"
]

ROLE_TECHNICAL = [
    "Testing the candidate's technical knowledge related to the job.",
    "Evaluating programming and software development knowledge.",
    "Assessing understanding of databases, APIs and frameworks.",
    "Testing knowledge of algorithms and data structures.",
    "Evaluating system design and software architecture understanding.",
    "Understanding debugging, testing and implementation skills.",
    "Explain the difference between REST and GraphQL.",
    "How does database indexing work?",
    "What is the difference between authentication and authorization?",
    "How would you design a scalable API?",
    "Explain how you would debug a memory leak."
]

CATEGORIES = {
    "WELCOME": WELCOME,
    "WARM_UP": WARM_UP,
    "EXPERIENCE": EXPERIENCE,
    "BEHAVIORAL": BEHAVIORAL,
    "SITUATIONAL": SITUATIONAL,
    "ROLE_TECHNICAL": ROLE_TECHNICAL,
}



In [11]:

rows = []
for label, texts in CATEGORIES.items():
    for t in texts:
        rows.append({"text": t, "label": label})

dataset = pd.DataFrame(rows)

In [12]:

# =========================================================
# 2. LABEL MAPPING
# =========================================================
labels = sorted(dataset["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

dataset["label"] = dataset["label"].map(label2id)

In [14]:

# =========================================================
# 3. SPLIT (stratified)
# =========================================================
train_df, val_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    stratify=dataset["label"],
)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

In [15]:
# =========================================================
# 4. TOKENIZER
# =========================================================
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    # dynamic padding is done later by DataCollatorWithPadding
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# Remove raw text column (not needed by model)
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])


Map: 100%|██████████| 13/13 [00:00<00:00, 1556.28 examples/s]


In [16]:
# =========================================================
# 5. MODEL
# =========================================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# =========================================================
# 6. METRICS
# =========================================================
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


In [18]:
# =========================================================
# 7. TRAINING ARGS
# =========================================================
training_args = TrainingArguments(
    output_dir="./question_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,            # more epochs...
    weight_decay=0.01,
    warmup_ratio=0.1,               # warmup for stability
    lr_scheduler_type="cosine",     # cosine decay
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,             # keep only best 2 checkpoints
    logging_steps=5,
    report_to="none",
    seed=42,
)

# =========================================================
# 8. TRAINER
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),  # dynamic padding
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop if no improvement
)

trainer.train()


C:\Users\rusha\AppData\Local\Temp\ipykernel_75572\3585390851.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
                                              
 10%|█         | 4/40 [00:01<00:10,  3.41it/s]

{'eval_loss': 1.7975138425827026, 'eval_accuracy': 0.15384615384615385, 'eval_f1_macro': 0.044444444444444446, 'eval_runtime': 0.1805, 'eval_samples_per_second': 72.024, 'eval_steps_per_second': 5.54, 'epoch': 1.0}


 15%|█▌        | 6/40 [00:04<00:25,  1.33it/s]

{'loss': 1.8149, 'grad_norm': 2.244140625, 'learning_rate': 2.9942920471376185e-05, 'epoch': 1.25}


                                              
 20%|██        | 8/40 [00:04<00:13,  2.33it/s]

{'eval_loss': 1.7843916416168213, 'eval_accuracy': 0.23076923076923078, 'eval_f1_macro': 0.1346153846153846, 'eval_runtime': 0.1305, 'eval_samples_per_second': 99.634, 'eval_steps_per_second': 7.664, 'epoch': 2.0}


... (29 outputs left out)

{'eval_loss': 1.3133716583251953, 'eval_accuracy': 0.6153846153846154, 'eval_f1_macro': 0.5777777777777777, 'eval_runtime': 0.0401, 'eval_samples_per_second': 324.25, 'eval_steps_per_second': 24.942, 'epoch': 10.0}


100%|██████████| 40/40 [00:38<00:00,  1.05it/s]

{'train_runtime': 38.2312, 'train_samples_per_second': 13.078, 'train_steps_per_second': 1.046, 'train_loss': 1.492062497138977, 'epoch': 10.0}


TrainOutput(global_step=40, training_loss=1.492062497138977, metrics={'train_runtime': 38.2312, 'train_samples_per_second': 13.078, 'train_steps_per_second': 1.046, 'total_flos': 4308598266624.0, 'train_loss': 1.492062497138977, 'epoch': 10.0})

In [19]:
# =========================================================
# 9. SAVE
# =========================================================
trainer.save_model("./question_classifier")
tokenizer.save_pretrained("./question_classifier")

print("\nTraining complete!")
print("Label map:", id2label)

# =========================================================
# 10. DETAILED EVALUATION ON VALIDATION SET
# =========================================================
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print("\nClassification report:")
print(
    classification_report(
        y_true, y_pred, target_names=[id2label[i] for i in range(len(labels))]
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))



Training complete!
Label map: {0: 'BEHAVIORAL', 1: 'EXPERIENCE', 2: 'ROLE_TECHNICAL', 3: 'SITUATIONAL', 4: 'WARM_UP', 5: 'WELCOME'}


100%|██████████| 1/1 [00:00<00:00, 36.26it/s]


Classification report:
                precision    recall  f1-score   support

    BEHAVIORAL       1.00      0.67      0.80         3
    EXPERIENCE       1.00      1.00      1.00         2
ROLE_TECHNICAL       1.00      0.50      0.67         2
   SITUATIONAL       1.00      1.00      1.00         2
       WARM_UP       0.33      1.00      0.50         2
       WELCOME       0.00      0.00      0.00         2

      accuracy                           0.69        13
     macro avg       0.72      0.69      0.66        13
  weighted avg       0.74      0.69      0.67        13


Confusion matrix:
[[2 0 0 0 1 0]
 [0 2 0 0 0 0]
 [0 0 1 0 1 0]
 [0 0 0 2 0 0]
 [0 0 0 0 2 0]
 [0 0 0 0 2 0]]



d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [21]:
# =========================================================
# 11. INFERENCE
# =========================================================
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./question_classifier",
    tokenizer="./question_classifier",
    device=-1,  # use 0 if you have a GPU
)

samples = [
    ("Tell me about a difficult project you worked on.", "EXPERIENCE"),
    ("What would you do if your teammate refused to cooperate?", "SITUATIONAL"),
    ("Introduce yourself and tell me about your background.", "WARM_UP"),
    ("How are you feeling today?", "WELCOME"),
    ("Tell me about a time you failed and how you recovered.", "BEHAVIORAL"),
    ("Imagine you are an interviewer, what question would you ask?", "SITUATIONAL"),
]

print("\nInference results:")
for text, expected in samples:
    result = classifier(text)[0]
    print(f"\n{text}")
    print(f"  Predicted : {result['label']:<15} ({result['score']:.3f})")
    print(f"  Expected  : {expected}")



Inference results:

Tell me about a difficult project you worked on.
  Predicted : EXPERIENCE      (0.349)
  Expected  : EXPERIENCE

What would you do if your teammate refused to cooperate?
  Predicted : SITUATIONAL     (0.347)
  Expected  : SITUATIONAL

Introduce yourself and tell me about your background.
  Predicted : WARM_UP         (0.324)
  Expected  : WARM_UP

How are you feeling today?
  Predicted : WELCOME         (0.285)
  Expected  : WELCOME

Tell me about a time you failed and how you recovered.
  Predicted : EXPERIENCE      (0.251)
  Expected  : BEHAVIORAL

Imagine you are an interviewer, what question would you ask?
  Predicted : SITUATIONAL     (0.234)
  Expected  : SITUATIONAL


### guess i ll do something easier

embeding for classification

In [30]:
import ollama
from sklearn.metrics.pairwise import cosine_similarity

In [132]:
def get_embedding(text):
    response = ollama.embeddings(model='nomic-embed-text:latest', prompt=text)
    return response['embedding']

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

In [135]:

# 1. Define the categories with a single, strong description for each
categories = {
    "WARM_UP": """Asking the candidate to introduce themselves.",
    "Understanding the candidate's personal and educational background.",
    "Learning about the candidate's interests and career aspirations.",
    "Understanding why the candidate applied for this position.",
    "Exploring the candidate's motivation and career goals.",
    "What are your greatest strengths?",
    "Why are you interested in this role?",
    "How would you describe yourself professionally?",
    "What motivates you in your career?",
    "What are you looking for in your next opportunity?"
    """,

    "EXPERIENCE": """"Exploring the candidate's previous professional experience.",
    "Understanding previous jobs, internships and professional responsibilities.",
    "Discussing projects the candidate has completed.",
    "Understanding the candidate's individual contributions to projects.",
    "Evaluating practical experience and ownership of responsibilities.",
    "What projects have you worked on?",
    "What was your role in your previous project?",
    "What responsibilities did you handle during your internship?",
    "What is your most significant professional achievement?",
    "What was the biggest challenge you faced while developing your project?",
    "Which technologies have you used in your previous work?"
    """,

    "BEHAVIORAL": """"Exploring how the candidate behaved in previous real-life situations.",
    "Evaluating teamwork and collaboration through past experiences.",
    "Understanding how the candidate handled workplace conflicts.",
    "Evaluating leadership and communication through previous examples.",
    "Understanding how the candidate responded to failure and criticism.",
    "Asking about situations involving pressure and tight deadlines.",
    "Tell me about a time you disagreed with a colleague.",
    "Describe a situation where you helped a struggling teammate.",
    "Tell me about a mistake you made and how you handled it.",
    "Describe a time you received negative feedback.",
    "Tell me about a time you had to take responsibility for a failure.",
    "Describe a situation where you demonstrated leadership."
    """,
    "SITUATIONAL" : """ "Presenting hypothetical workplace situations to evaluate decision-making.",
    "Evaluating how the candidate would respond to unexpected challenges.",
    "Understanding how the candidate would resolve hypothetical conflicts.",
    "Assessing problem-solving and adaptability in unfamiliar situations.",
    "Evaluating prioritization when facing competing responsibilities.",
    "What would you do if your teammate refused to cooperate?",
    "How would you handle multiple urgent deadlines?",
    "Imagine you are assigned a task you have never performed before. What would you do?",
    "What would you do if you disagreed with your manager?",
    "How would you respond if a project suddenly failed?",
    "Suppose you discovered a serious mistake just before delivery. How would you handle it?"
"""
}

# 2. Load a fast, lightweight model

# 3. Embed the category descriptions
cat_keys = list(categories.keys())
cat_texts = list(categories.values())

In [136]:
cat_embeddings = []
for x in cat_texts:
    cat_embeddings.append(get_embedding(x))

In [137]:
cat_embeddings[1]

[-0.4422038793563843,
 -0.1358453333377838,
 -2.2856318950653076,
 0.1022314578294754,
 0.8480405807495117,
 0.6720061302185059,
 0.573917031288147,
 -0.0187351256608963,
 0.540984570980072,
 0.1913120448589325,
 -0.5333147644996643,
 -0.3242124617099762,
 0.9166779518127441,
 1.1748143434524536,
 -0.16440773010253906,
 -0.5267206430435181,
 0.7390444874763489,
 -2.099471092224121,
 -0.41606634855270386,
 -0.0362994559109211,
 0.16530682146549225,
 -2.158745288848877,
 -0.406442254781723,
 0.7695509195327759,
 0.5392802953720093,
 0.6308379769325256,
 -0.7155674695968628,
 -0.7023293375968933,
 -0.9255340695381165,
 -1.1455790996551514,
 0.4414505958557129,
 -0.335406094789505,
 0.18116044998168945,
 -0.3973628282546997,
 -0.6061267852783203,
 -1.5630074739456177,
 0.9210528135299683,
 0.7578997611999512,
 -0.06084541976451874,
 -0.4797620177268982,
 1.0820430517196655,
 -0.30603331327438354,
 0.4702954590320587,
 -2.1447935104370117,
 0.9142919182777405,
 -0.2931130528450012,
 0.89796

In [138]:
# 4. Test function
def classify_question(user_question):
    # Embed the user's question
    q_embedding = get_embedding(user_question)
    
    # Calculate cosine similarity against all categories
    similarities = util.cos_sim(q_embedding, cat_embeddings)[0]
    # similarities = cosine_similarity(q_embedding, cat_embeddings)[0]
    print(similarities)
    # Get the best match
    best_idx = torch.argmax(similarities).item()
    best_category = cat_keys[best_idx]
    score = similarities[best_idx].item()
    
    print(f"Q: {user_question}")
    print(f"👉 Category: {best_category} (Confidence: {score:.2f})\n")

In [139]:
q="imagine you are the one who is interviewing then what question would you ask me"

In [140]:
classify_question(q)

tensor([0.6516, 0.6294, 0.6012, 0.6349])
Q: imagine you are the one who is interviewing then what question would you ask me
👉 Category: WARM_UP (Confidence: 0.65)



In [87]:

# 5. Test it
test_questions = [
    "Tell me about yourself.",
    "What was your role in your last project?",
    "Describe a time you disagreed with a teammate.",
    "Why do you want to work here?",
    "What technologies did you use during your internship?",
    "lets say you are the one who is interviewing then what question would you ask me"
]

for q in test_questions:
    classify_question(q)

Q: Tell me about yourself.
👉 Category: BEHAVIORAL (Confidence: 0.24)

Q: What was your role in your last project?
👉 Category: EXPERIENCE (Confidence: 0.39)

Q: Describe a time you disagreed with a teammate.
👉 Category: BEHAVIORAL (Confidence: 0.40)

Q: Why do you want to work here?
👉 Category: WARM_UP (Confidence: 0.28)

Q: What technologies did you use during your internship?
👉 Category: EXPERIENCE (Confidence: 0.38)

Q: lets say you are the one who is interviewing then what question would you ask me
👉 Category: WARM_UP (Confidence: 0.51)

